# Code pour le calcul des indicateurs suivi par le PNLP

**Auteur** : Alex Kaldjian et Marie-Julie Lambert et Meschac Mutombo 

**Date** : Janvier 2021

**Projet** : EOC Malaria (Gates)

**Pays** : RDC

Contenu : Ce notebook contient le code permettant de calculer les différents indicateurs suivis par le PNLP ainsi que leur classe ou score. Les résultats de ce travail seront ensuite chargés directement dans Tableau pour une visualisation optimale des résultats.

## 1. Charger les librairies, faire la connection avec la base de données Postegres et charger les données

La première étape est de charger l'ensemble des librairies nécessaires pour la réalisation des calculs d'indicateurs. Ensuite, nous établirons la connexion avec la base de données PostgreSQL qui pourra être liée directement à Tableau. Enfin, nous chargerons les données qui viennent directement de DHIS2 et nous les visualiserons.

In [ ]:
suppressWarnings(install.packages("zoo"))

In [ ]:
suppressWarnings(library(zoo, warn.conflicts = FALSE))
suppressWarnings(library(jsonlite))
suppressWarnings(library(haven, warn.conflicts = FALSE))
suppressWarnings(library(dplyr))
suppressWarnings(library(tidyr))
suppressWarnings(library(data.table, warn.conflicts = FALSE))
library(arrow);

suppressWarnings(options(dplyr.summarise.inform = FALSE))
library(RPostgres); 

In [ ]:
# Parameters
ANNEE_A_ANALYSER <- 2026
UPLOAD <- FALSE

In [ ]:
# paths
PROJECT_BASE_PATH <- '/home/hexa/workspace/pnlp-tdb-pipeline/'
RAW_DATA_PATH <- paste0(PROJECT_BASE_PATH, "data/raw/")
INTERIM_DATA_PATH <- paste0(PROJECT_BASE_PATH, "data/interim/")
OUTPUT_DATA_PATH <- paste0(PROJECT_BASE_PATH, "data/processed_extra_indicators/")

In [ ]:
# CHECK THE MOST UPDATED DATA ---> READ pnlp-tdb-pipeline/data/raw/population/note.txt
pop <- fread(paste0(RAW_DATA_PATH,
                       "population/",
                       "analytics.csv"))

names(pop) <- gsub('DX', 'DE', toupper(names(pop)))

pop <- pop %>% filter(PERIOD == ANNEE_A_ANALYSER)

In [ ]:
df <- read_feather(paste0(INTERIM_DATA_PATH, 
                          "PNLP_indicators_", 
                          ANNEE_A_ANALYSER,
                          ".feather"))

ind <- read_feather(paste0(INTERIM_DATA_PATH, 
                          "all_cause_mortality_", 
                          ANNEE_A_ANALYSER,
                          ".feather"))

cp <- read_feather(paste0(INTERIM_DATA_PATH, 
                          "PNLP_DHIS_reporting_rates_", 
                          ANNEE_A_ANALYSER,
                          ".feather"))

In [ ]:
# head(df)
# unique(df[df$DE_UID == "pak21wvkWJC" ,]$PERIOD)

### Fonctions utilisées

In [ ]:
source(paste0(PROJECT_BASE_PATH, "helpers_indicators.R"))

### Préparation des données

In [ ]:
df_mensuel <- format_time_period(df, 'mois')
df_annuel <- format_time_period(df, 'annee')

In [ ]:
# free memory: raw data no longer needed
rm(df); gc()

In [ ]:
if (!is.null(ind)) {
    ind_mensuel <- format_time_period(ind, 'mois', data_type='IND')
}

In [ ]:
if (!is.null(cp)) {
    cp_mensuel <- format_time_period(cp, 'mois', data_type='CP')
}

In [ ]:
if (!is.null(ind)) {
    ind_mensuel <- ind_mensuel %>% mutate(COC_NAME = NA,
                                          DE_UID = IND_UID,
                                          OU_NAME = LEVEL_5_NAME) %>%

                                select(OU_UID, OU_NAME, Annee, 
                                       Filtre_time, COC_NAME,
                                       LEVEL_5_NAME, LEVEL_5_UID,
                                       LEVEL_4_NAME, LEVEL_4_UID, 
                                       LEVEL_3_NAME, LEVEL_3_UID, 
                                       LEVEL_2_NAME, LEVEL_2_UID, 
                                       LEVEL_1_NAME, LEVEL_1_UID, 
                                       Agg_period, DE_UID, value)
}

In [ ]:
pop_df <- pop %>% 
            mutate(Annee = PERIOD, Filtre_time = 'Année',
                   Agg_period = PERIOD, value = VALUE,
                   LEVEL_5_NAME = NA, LEVEL_5_UID = NA,
                   OU_NAME = LEVEL_4_NAME,
                   OU_UID = LEVEL_4_UID) %>%

            select(OU_NAME, OU_UID, Annee, 
                   Filtre_time, COC_NAME,
                   LEVEL_5_NAME, LEVEL_5_UID,
                   LEVEL_4_NAME, LEVEL_4_UID, 
                   LEVEL_3_NAME, LEVEL_3_UID, 
                   LEVEL_2_NAME, LEVEL_2_UID, 
                   LEVEL_1_NAME, LEVEL_1_UID, 
                   Agg_period, DE_UID, value)

annuel_avec_pop = rbind(df_annuel, pop_df) 

## 4. Calcul des indicateurs

### 4.1. Proportion de cas suspect testés

**Dénominateur :** 

Élément de données > A Paludisme > A1.4. Cas suspect (DENOM) -> `aZwnLALknnj`

+

B. Prise en charge au site de soin communautaire > B 10. 2. Cas de fièvre dans une zone à risque du paludisme (appellation au site de soin communautaire) -> `D3h3Qvl0332`

**Numérateur :**

B10.2. TDR réalisés Cas de fièvre -> `uIqmxJcTuub`

+

A1.4. TDR réalisé -> `CIzQAR8IWH1`

+

Goutte épaisse  (Indicateurs > Malaria > Cas suspect testé par microscopie)> D 5.3 GE réalisé - Examens réalisés -> `sRbXNrdKvyl`

In [ ]:
I1_nom <- 'Cas_Suspects_Teste'
I1_numerateur <- c('uIqmxJcTuub', 'CIzQAR8IWH1', 'sRbXNrdKvyl')
I1_denominateur <- c('aZwnLALknnj', 'D3h3Qvl0332')

In [ ]:
I1 <- calculer_indicateur(df_mensuel,
                          indicateur_name=I1_nom, 
                          numerator=I1_numerateur, 
                          denominator=I1_denominateur)

### 4.2. Taux de positivité au TDR et TDR / Goutte épaisse
Attention faire d’un côté TDR seul et de l’autre TDR et goutte épaisse 

**Dénominateur :**

B10.2. TDR réalisés Cas de fièvre -> `uIqmxJcTuub`

+

A1.4. TDR réalisé -> `CIzQAR8IWH1`

+

(quand on tient compte de la goutte épaisse)

(Indicateurs > Malaria > Cas suspect testé par microscopie > D 5.3 GE réalisé - Examens réalisés -> `sRbXNrdKvyl`) 

**Numérateur :** 

A1.4. TDR positif -> `SpmQSLRPMl4`

+

B 10.2. Dont positifs - Cas de fièvre (Prise en charge au site de soin Communautaire) -> `IPnqGkUUN6U`

+

(quand on tient compte de la goutte épaisse)

(Indicateurs > Malaria > PALU Cas de paludisme confirmé par microscopie*** > D 5.3 GE réalisé dont GE positive - Examens positifs) -> `e6qMP9fVtG9`

#### TDR Seul (2a)

In [ ]:
I2a_nom <- 'Positivite_TDR'
I2a_numerateur <- c('SpmQSLRPMl4', 'IPnqGkUUN6U')
I2a_denominateur <- c('uIqmxJcTuub', 'CIzQAR8IWH1')

In [ ]:
I2a <- calculer_indicateur(df_mensuel,
                           indicateur_name=I2a_nom,
                           numerator=I2a_numerateur,
                           denominator=I2a_denominateur)

#### 2b TDR + goutte épaisse

In [ ]:
I2b_nom <- 'Positivite_TDR_GE'
I2b_numerateur <- c('SpmQSLRPMl4', 'IPnqGkUUN6U', 'e6qMP9fVtG9')
I2b_denominateur <- c('uIqmxJcTuub', 'CIzQAR8IWH1', 'sRbXNrdKvyl')

In [ ]:
I2b <- calculer_indicateur(df_mensuel,
                           indicateur_name=I2b_nom,
                           numerator=I2b_numerateur,
                           denominator=I2b_denominateur)

### 4.3. Proportion de cas confirmé traité selon la politique nationale

**Dénominateur :**

A1.4. TDR positif -> `SpmQSLRPMl4`

+

B 10.2. Dont positifs - Cas de fièvre (Prise en charge au site de soin Communautaire) -> `IPnqGkUUN6U`

+

D 5.3 GE réalisé dont GE positive - Examens positifs -> `e6qMP9fVtG9`

**Numérateur :** 

Élément de données > B. Prise en charge au site de soins communautaires > B 10.2. Dont traités selon la politique nationale - Cas de fièvre -> `BCcFZzgTlxQ`

+

A 1.4. Paludisme grave traité -> `CGZbvJchfjk`

+

A1.4. Paludisme simple confirmé traité PN -> `nRm30I4w9En`

In [ ]:
I3_nom <- 'Cas_Confirme_Traité'
I3_numerateur <- c('BCcFZzgTlxQ', 'CGZbvJchfjk', 'nRm30I4w9En')
I3_denominateur <- c('SpmQSLRPMl4', 'IPnqGkUUN6U', 'e6qMP9fVtG9')

In [ ]:
I3 <- calculer_indicateur(df_mensuel,
                          indicateur_name=I3_nom,
                          numerator=I3_numerateur,
                          denominator=I3_denominateur)

### 4.4. Proportion des femmes enceintes ayant bénéficiées d’une MILD à la première CPN 1

Au site de soin communautaire, on ne fait pas de CPN. Donc on peut aller directement dans 

Indicateurs > PNLP_Paludisme > PNLP_Proportion des femmes enceintes ayant reçu la MILD à la CPN1

**Numérateur :**

A 2.1 MILD distribués a la CPN1 -> `uV53nh3MrYl`

**Dénominateur :**

A 2.1 CPN 1 -> `kGmdIbd3mbn`

In [ ]:
I4_nom <- 'FE_Enceinte_MILD'
I4_numerateur <- c('uV53nh3MrYl')
I4_denominateur <- c('kGmdIbd3mbn')

In [ ]:
I4 <- calculer_indicateur(df_mensuel,
                          indicateur_name=I4_nom,
                          numerator=I4_numerateur,
                          denominator=I4_denominateur)

### 4.5. Létalité hospitalière 

**Dénominateur :**

A1.4. Cas de paludisme grave -> `AxJhIi7tUam`

**Numérateur :**

B. Décès > B7.2. Paludisme Grave Décès -> `wfmDVt6RVm2`

+

D. Mortalité enregistrée en hospitalisation > Paludisme dcd -> `nFjuJLLeejV`

In [ ]:
I5_nom <- 'Letalite_Hospi'
I5_numerateur <- c('wfmDVt6RVm2', 'nFjuJLLeejV')
I5_denominateur <- c('AxJhIi7tUam')

In [ ]:
I5 <- calculer_indicateur(df_mensuel,
                          indicateur_name=I5_nom,
                          numerator=I5_numerateur,
                          denominator=I5_denominateur)

### 4.6. Proportion d'enfants de moins de un an ayant reçu une MILD lors de la CPS

**Dénominateur :**

VAR 0 et 11 mois Avancé --> `cTLKwfG8pSv` Avec les 3 COC : Avancé `KPDzIsWq7JK` , Fixe `NOHlOxLczjc` et Mobile `Dr4rWTqepnP`
    
**After 2026-01 (for now):**  
B 8.4 VAR 1/RR 1 --> `pak21wvkWJC`  
Avec 3 COC:  
-0 et 11 mois, Avancé `KPDzIsWq7JK`  
-0 et 11 mois, Fixe `dqydGQFHahb`  
-0 et 11 mois, Mobile `oSYTyzezWif`    
  
**Numérateur :**

B - Consultation Préscolaire > MILD distribuées 6 et 11 mois ->  `QafSfDEL1Ku` Mais attention, COC uniquement 6 à 11 mois : `Dl5DTT7zYLV`


In [ ]:
I6_nom <- 'MILD_CPS'
I6_numerateur <- c('QafSfDEL1Ku')

# Note: Data element "cTLKwfG8pSv" not being used (filled in DHIS2 since ~2025)
# If we want to update this indicator for 2025, we need to re-run the process.
# Data is already available.
if (unique(df_mensuel$Annee)[1] >= "2026") {
    I6_denominateur <- c('pak21wvkWJC')
} else {
    I6_denominateur <- c('cTLKwfG8pSv')
}

In [ ]:
I6 <- calculer_indicateur(df_mensuel,
                          indicateur_name=I6_nom,
                          numerator=I6_numerateur,
                          denominator=I6_denominateur)

### 4.7. % de femmes enceintes ayant bénéficiés de SP2 lors de la CPN

**Dénominateur :**
A Consultation prenatale > CPN 1 (A 2.1.) --> `kGmdIbd3mbn` 

**Numérateur :**

A2.1. Sulfadox + Pyrimét 2ème dos reçue  ->  `fsBcCCKHGUx` 

In [ ]:
I7_nom <- 'Proportion_SP2'
I7_numerateur <- c('fsBcCCKHGUx')
I7_denominateur <- c('kGmdIbd3mbn')

In [ ]:
I7 <- calculer_indicateur(df_mensuel,
                          indicateur_name=I7_nom,
                          numerator=I7_numerateur,
                          denominator=I7_denominateur)

### 4.8. % de femmes enceintes ayant bénéficiés de SP3 lors de la CPN

**Dénominateur :**
A Consultation prenatale > CPN 1 (A 2.1.) --> `kGmdIbd3mbn` 

**Numérateur :**

A2.1. Sulfadox + Pyrimét 3ème dos reçue  ->  `pOZcn5GNnRD` 

In [ ]:
I8_nom <- 'Proportion_SP3'
I8_numerateur <- c('pOZcn5GNnRD')
I8_denominateur <- c('kGmdIbd3mbn')

In [ ]:
I8 <- calculer_indicateur(df_mensuel,
                          indicateur_name=I8_nom,
                          numerator=I8_numerateur,
                          denominator=I8_denominateur)

### 4.9. Incidence annuelle

#### !! **A calculer uniquement annuellement** !!

**Dénominateur :**
Population de l’aire de santé --> `WLSKVyA8LoY` 

**Numérateur :**

B. Prise en Charge au site de soins communautaires > Dont Positifs – Cas de fièvre (B 10.2)   ->  `IPnqGkUUN6U` 

 +

A 1.4. Paludisme grave --> `AxJhIi7tUam`

 +
 
A1.4 Paludisme simple confirmé --> `rfeqp2kdOGi`




In [ ]:
I9_nom <- 'Incidence_Annuelle'
I9_numerateur <- c('IPnqGkUUN6U', 'AxJhIi7tUam', 'rfeqp2kdOGi')
I9_denominateur <- c('WLSKVyA8LoY')

In [ ]:
I9 <- calculer_indicateur(annuel_avec_pop,
                           indicateur_name=I9_nom, 
                           numerator=I9_numerateur, 
                           denominator=I9_denominateur,
                           population=TRUE)

### 4.10. Taux de mortalité

#### !! **A calculer uniquement annuellement** !!

**Dénominateur :**
Population de l’aire de santé --> `WLSKVyA8LoY` 

**Numérateur :**

B. Décès > B7.2. Paludisme Grave Décès ->  `wfmDVt6RVm2` 

+ 

D Mortalité enregistrée en hospi > D 9.6. Paludisme dcd  --> `nFjuJLLeejV`


In [ ]:
I10_nom <- 'Taux_Mortalite'
I10_numerateur <- c('wfmDVt6RVm2', 'nFjuJLLeejV')
I10_denominateur <- c('WLSKVyA8LoY')

In [ ]:
I10 <- calculer_indicateur(annuel_avec_pop,
                           indicateur_name=I10_nom, 
                           numerator=I10_numerateur, 
                           denominator=I10_denominateur, 
                           population=TRUE)

In [ ]:
rm(df_annuel, annuel_avec_pop, pop, pop_df); gc()


### 4.11. Mortalité proportionnelle

**Dénominateur :**
PALU – Tous les décès --> `fvlFcxuGRng` 

**Numérateur :**

B. Décès > B7.2. Paludisme Grave Décès ->  `wfmDVt6RVm2` 

+ 

D Mortalité enregistrée en hospi > D 9.6. Paludisme dcd  --> `nFjuJLLeejV`

In [ ]:
if (!is.null(ind)) {
    ind_mensuel <- rbind(filter(df_mensuel, 
                            DE_UID %in% c('wfmDVt6RVm2', 'nFjuJLLeejV')), ind_mensuel)
}

In [ ]:
I11_nom <- 'Mortalite_Prop'
I11_numerateur <- c('wfmDVt6RVm2', 'nFjuJLLeejV')
I11_denominateur <- c('fvlFcxuGRng')

In [ ]:
if (!is.null(ind)) {
    I11 <- calculer_indicateur(ind_mensuel, 
                               indicateur_name=I11_nom,
                               numerator = I11_numerateur,
                               denominator = I11_denominateur)
}

### Promptitude et complétude

A. Services de base -> `pMbC0FJPkcm`  
B. Services secondaires -> `maDtHIFrSHx`  
D. Service hôpital -> `OeWrFwkFMvf`

In [ ]:
service_base_uid <- 'pMbC0FJPkcm'
service_second_uid <- 'maDtHIFrSHx'
service_hospital_uid <- 'OeWrFwkFMvf'

reporting_type_comp <- 'REPORTING_RATE'
reporting_type_promp <- 'REPORTING_RATE_ON_TIME'

### 4.12. Services de base – Complétude


In [ ]:
I12_nom <- 'DHIS_comp_service_base'

In [ ]:
if(!is.null(cp)) {
    I12 <- aggreger_reporting(cp_mensuel,
                              reporting_type=reporting_type_comp, 
                              reporting_uid=service_base_uid, 
                              indicateur_name=I12_nom)
}

### 4.13. Services de base – Promptitude


In [ ]:
I13_nom <- 'DHIS_prompt_service_base'

In [ ]:
if (!is.null(cp)) {
    I13 <- aggreger_reporting(cp_mensuel,
                          reporting_type=reporting_type_promp, 
                          reporting_uid=service_base_uid, 
                          indicateur_name=I13_nom)
}

### 4.14. Services secondaires – Complétude


In [ ]:
I14_nom <- 'DHIS_comp_service_second'

In [ ]:
if(!is.null(cp)) {
    I14 <- aggreger_reporting(cp_mensuel,
                      reporting_type=reporting_type_comp, 
                      reporting_uid=service_second_uid, 
                      indicateur_name=I14_nom)
}

### 4.15. Services secondaires – Promptitude


In [ ]:
I15_nom <- 'DHIS_prompt_service_second'

In [ ]:
if (!is.null(cp)) {
    I15 <- aggreger_reporting(cp_mensuel,
                      reporting_type=reporting_type_promp, 
                      reporting_uid=service_second_uid, 
                      indicateur_name=I15_nom)
}

### 4.16. Services hôpital – Complétude


In [ ]:
I16_nom <- 'DHIS_comp_service_hosp'

In [ ]:
if(!is.null(cp)) {
   I16 <- aggreger_reporting(cp_mensuel,
                      reporting_type=reporting_type_comp, 
                      reporting_uid=service_hospital_uid, 
                      indicateur_name=I16_nom)
}

### 4.17. Services hôpital – Promptitude


In [ ]:
I17_nom <- 'DHIS_prompt_service_hosp'

In [ ]:
if (!is.null(cp)) {    
   I17 <- aggreger_reporting(cp_mensuel,
                      reporting_type=reporting_type_promp, 
                      reporting_uid=service_hospital_uid, 
                      indicateur_name=I17_nom)
}

### 4.18. Total cas du paludisme

**Dénominateur :**
1

**Numérateur :**

B. Prise en Charge au site de soins communautaires > Dont Positifs – Cas de fièvre (B 10.2)   ->  `IPnqGkUUN6U` 

 +

A 1.4. Paludisme grave --> `AxJhIi7tUam`

 +
 
A1.4 Paludisme simple confirmé --> `rfeqp2kdOGi`




In [ ]:
I18_nom <- 'Total_Cas_Palu'
I18_IDs <- c('IPnqGkUUN6U', 'AxJhIi7tUam', 'rfeqp2kdOGi')

In [ ]:
I18 <- sommer_indicateur(df_mensuel,
                         indicateur_name=I18_nom, 
                         element_ids=I18_IDs)

### 4.19. Nombre de TDR Réalises 

**Dénominateur :**
1

**Numérateur :**
Nombre de TDR Réalises --> `CIzQAR8IWH1`



In [ ]:
I19_nom <- 'TDRs_réalisés'
I19_IDs <- c('CIzQAR8IWH1')

In [ ]:
I19 <- sommer_indicateur(df_mensuel,
                         indicateur_name=I19_nom, 
                         element_ids=I19_IDs)

### 4.20. Nombre de Cas de paludisme simple confirmé

**Dénominateur :**
1

**Numérateur :**
Nombre de Cas de paludisme simple confirmé --> `rfeqp2kdOGi`

In [ ]:
I20_nom <- 'Cas_simples_confirmés'
I20_IDs <- c('rfeqp2kdOGi')

In [ ]:
I20 <- sommer_indicateur(df_mensuel,
                         indicateur_name=I20_nom, 
                         element_ids=I20_IDs)

### 4.21. Nombre de Cas de paludisme simple confirmé traité selon le PN  

**Dénominateur :**
1

**Numérateur :**
Nombre de Cas de paludisme simple confirmé traité selon le PN --> `nRm30I4w9En`



In [ ]:
I21_nom <- 'Cas_simples_traité_PN'
I21_IDs <- c('nRm30I4w9En')

In [ ]:
I21 <- sommer_indicateur(df_mensuel,
                         indicateur_name=I21_nom, 
                         element_ids=I21_IDs)

### 4.22. Nombre de Cas de paludisme grave 

**Dénominateur :**
1

**Numérateur :**
Nombre de Cas de paludisme grave --> `AxJhIi7tUam`



In [ ]:
I22_nom <- 'Cas_graves_confirmés'
I22_IDs <- c('AxJhIi7tUam')

In [ ]:
I22 <- sommer_indicateur(df_mensuel,
                         indicateur_name=I22_nom, 
                         element_ids=I22_IDs)

### 4.23. Nombre de Cas de paludisme grave traité

**Dénominateur :**
1

**Numérateur :**
Nombre de Cas de paludisme grave traité --> `CGZbvJchfjk`



In [ ]:
I23_nom <- 'Cas_graves_traités'
I23_IDs <- c('CGZbvJchfjk')

In [ ]:
I23 <- sommer_indicateur(df_mensuel,
                         indicateur_name=I23_nom, 
                         element_ids=I23_IDs)

### 4.24. Nombre de Cas de paludisme présumé 

**Dénominateur :**
1

**Numérateur :**
Nombre de Cas de paludisme présumé --> `aK0QXqm8Zxn`



In [ ]:
I24_nom <- 'Cas_présumés'
I24_IDs <- c('aK0QXqm8Zxn')

In [ ]:
I24 <- sommer_indicateur(df_mensuel,
                         indicateur_name=I24_nom, 
                         element_ids=I24_IDs)

### 4.25. Nombre de Cas de paludisme présumé traité

**Dénominateur :**
1

**Numérateur :**
Nombre de Cas de paludisme présumé traité  --> `JXm8J8GRJxI`



In [ ]:
I25_nom <- 'Cas_présumés_traités'
I25_IDs <- c('JXm8J8GRJxI')

In [ ]:
I25 <- sommer_indicateur(df_mensuel,
                         indicateur_name=I25_nom, 
                         element_ids=I25_IDs)

### 4.26 Nombre de paludisme grave décès (* NEW)

**Dénominateur :**
1

**Numérateur :**
B. Décès > B7.2. Paludisme Grave Décès  --> `wfmDVt6RVm2`



In [ ]:
I26_nom <- 'Palu_grave_deces'
I26_IDs <- c('wfmDVt6RVm2')

In [ ]:
I26 <- sommer_indicateur(df_mensuel,
                         indicateur_name=I26_nom, 
                         element_ids=I26_IDs)

### 4.27 Nombre de décès enregistrée en hospitalisation (* NEW)

**Dénominateur :**
1

**Numérateur :**
D Mortalité enregistrée en hospi > D 9.6. Paludisme dcd --> `nFjuJLLeejV`

In [ ]:
I27_nom <- 'Mortal_hosp_deces'
I27_IDs <- c('nFjuJLLeejV')

In [ ]:
I27 <- sommer_indicateur(df_mensuel,
                         indicateur_name=I27_nom, 
                         element_ids=I27_IDs)

In [ ]:
# free memory: FOSA level data no longer needed once all indicators are computed
rm(df_mensuel, ind, ind_mensuel, cp, cp_mensuel)
gc()


## Stockage des résultats

In [ ]:
I_list <- list(I1, I2a, I2b, I3, I4, I5, 
               I6, I7, I8, I9, I10, I11, I12, 
               I13, I14, I15, I16, I17, I18,
               I19, I20, I21, I22, I23, I24, I25,
               I26, I27) ## extra indicators 26, 27

output_final <- do.call(rbind, I_list)

In [ ]:
dim(output_final)

In [ ]:
# Write year file
write_feather(output_final, paste0(OUTPUT_DATA_PATH, 
                    "PNLP_dash_indicators_", 
                    ANNEE_A_ANALYSER,
                    ".feather"))

In [ ]:
# free memory: results are saved to feather
rm(I_list, output_final,
   I1, I2a, I2b, I3, I4, I5,
   I6, I7, I8, I9, I10, I11, I12,
   I13, I14, I15, I16, I17, I18,
   I19, I20, I21, I22, I23, I24, I25,
   I26, I27)
gc()

## 5. Création de la BDD de sortie

In [ ]:
# if (UPLOAD) {
    
#     #####
#     # test to see if clearing memory of largest 
#     # objects allows process to run
#     rm(df_mensuel, df, output_final)
#     gc()
#     #####
    
#     file.names <- list.files(OUTPUT_DATA_PATH, pattern = 'indicators.+feather')
#     file.paths <- paste0(OUTPUT_DATA_PATH, file.names)
    
#     output.data <- lapply(file.paths, read_feather)
#     output.data <- lapply(output.data, setDT)
    
#     output.combined <- rbindlist(output.data)
    
# }


In [ ]:
# if (UPLOAD) {
    
#     setDT(output.combined)

#     # Yearly computed indicators -> Agg_period is equel to YEAR -> we just set those to "Jan" + YEAR
#     output.combined[Agg_period == Annee, Agg_period := paste('Jan', Annee)]
    
#     dbWriteTable(con, "indicators_ND", output.combined, overwrite=TRUE)
# }

In [ ]:
if (UPLOAD) {

    # reconnect: the connection opened at the start may have timed out during the computation
    con <- dbConnect(
        RPostgres::Postgres(),
        dbname = Sys.getenv("WORKSPACE_DATABASE_DB_NAME"),
        host = Sys.getenv("WORKSPACE_DATABASE_HOST"),
        port = Sys.getenv("WORKSPACE_DATABASE_PORT"),
        user = Sys.getenv("WORKSPACE_DATABASE_USERNAME"),
        password = Sys.getenv("WORKSPACE_DATABASE_PASSWORD")
    )

    file_path <- file.path(OUTPUT_DATA_PATH, paste0("PNLP_dash_indicators_", ANNEE_A_ANALYSER, ".feather")) # 1 file CURRENT YEAR
    if (!file.exists(file_path)) stop("Feather file not found: ", file_path)

    chunk_size <- 500000

    message("Processing: ", file_path)
    dt <- read_feather(file_path)
    setDT(dt)

    # Yearly computed indicators -> Agg_period is equal to YEAR -> we just set those to "Jan" + YEAR
    dt[Agg_period == Annee, Agg_period := paste('Jan', Annee)]

    total_rows <- nrow(dt)
    num_chunks <- ceiling(total_rows / chunk_size)

    # Delete the current YEAR and append the new data in a single transaction:
    # if any step fails, the table keeps its previous content
    dbWithTransaction(con, {
        if (dbExistsTable(con, "indicators_ND")) {
            n_deleted <- dbExecute(
                con,
                sprintf('DELETE FROM public."indicators_ND" WHERE "Annee" = %d', ANNEE_A_ANALYSER)
            )
            message("Rows deleted for year ", ANNEE_A_ANALYSER, ": ", n_deleted)
        }

        for (i in seq_len(num_chunks)) {
            start_row <- (i - 1) * chunk_size + 1
            end_row <- min(i * chunk_size, total_rows)
            chunk <- dt[start_row:end_row]

            dbWriteTable(
                con,
                name = "indicators_ND",
                value = chunk,
                append = TRUE,
                row.names = FALSE
            )
            message("  Uploaded chunk ", i, "/", num_chunks)
            flush.console()
        }
    })

    rm(dt, chunk)
    gc()
    dbDisconnect(con)
}

**The following code was used to revert the DB table until December 2024.**

In [ ]:
# Revert to 2024...

# Select the files to merge
# file.names <- c("PNLP_dash_indicators_2018.feather", "PNLP_dash_indicators_2019.feather",
#                 "PNLP_dash_indicators_2020.feather", "PNLP_dash_indicators_2021.feather",
#                 "PNLP_dash_indicators_2022.feather", "PNLP_dash_indicators_2023.feather",
#                 "PNLP_dash_indicators_2024.feather")
# file.paths <- paste0(OUTPUT_DATA_PATH, file.names)
# output.data <- lapply(file.paths, read_feather)
# output.data <- lapply(output.data, setDT)
# output.combined <- rbindlist(output.data)

# Set as table
# setDT(output.combined)

# Fix the Agg_period column..
# # Yearly computed indicators -> Agg_period is equel to YEAR -> we just set those to "Jan" + YEAR
# output.combined[Agg_period == Annee, Agg_period := paste('Jan', Annee)]

# push to the DB
# dbWriteTable(con, "indicators_ND", output.combined, overwrite=TRUE)